<a href="https://colab.research.google.com/github/Xv-123/dl/blob/main/templates/11_sliding_window.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/11_sliding_window.ipynb)

# 🔴 Hard: Sliding Window Attention

Implement **Sliding Window Attention** — used in Longformer, Mistral, etc. for efficient long-context processing.

Each position $i$ can only attend to positions $j$ where $|i - j| \le w$ (the window size).

### Signature
```python
def sliding_window_attention(Q, K, V, window_size):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
    # window_size: int — position i attends to [i-w, i+w]
```

### Rules
- Do **NOT** use sparse attention libraries
- Mask positions outside the window with `-inf`
- `window_size=0`: only self — output should equal V
- `window_size >= seq_len`: equivalent to full attention

In [73]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [74]:
import torch
import math

In [75]:
# # ✏️ YOUR IMPLEMENTATION HERE

# def sliding_window_attention(Q, K, V, window_size):
#     pass  # Replace this

In [80]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def transpose_qkv(X, num_heads):
    # X: [batch, seq, d_model]
    X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)
    X = X.permute(0, 2, 1, 3)
    return X.reshape(-1, X.shape[2], X.shape[3])

def transpose_output(X, num_heads):
    # X: [batch*heads, seq, d_k]
    X = X.reshape(-1, num_heads, X.shape[1], X.shape[2])
    X = X.permute(0, 2, 1, 3)
    return X.reshape(X.shape[0], X.shape[1], -1)


class SlidingWindowDotProductAttention(nn.Module):
    def __init__(self, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.attention_weights = None

    def forward(self, queries, keys, values, window_size):
        bh, seq_q, dk = queries.shape
        _, seq_k, _ = keys.shape
        out = torch.zeros((bh, seq_q, dk), device=queries.device, dtype=queries.dtype)

        # 缩放因子
        scale = torch.sqrt(torch.tensor(dk, dtype=torch.float32))

        for i in range(seq_q):
            left = max(0, i - window_size)
            right = min(seq_k, i + window_size + 1)
            k_win = keys[:, left:right, :]
            v_win = values[:, left:right, :]
            q_i = queries[:, i:i+1, :]

            score_win = torch.matmul(q_i, k_win.transpose(-1, -2)) / scale
            attn_win = F.softmax(score_win, dim=-1)
            attn_win = self.dropout(attn_win)

            out[:, i:i+1, :] = torch.matmul(attn_win, v_win)

        self.attention_weights = None
        return out

In [81]:
import torch
import torch.nn as nn
import torch.nn.functional as F



class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, d_model=8, dropout=0.0, bias=False, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.num_hiddens = d_model
        self.attention = SlidingWindowDotProductAttention(dropout)

        self.W_q = nn.Linear(d_model, d_model, bias=bias)
        self.W_k = nn.Linear(d_model, d_model, bias=bias)
        self.W_v = nn.Linear(d_model, d_model, bias=bias)
        self.W_o = nn.Linear(d_model, d_model, bias=bias)

    def forward(self, queries, keys, values, window_size=10000):
        queries = self.W_q(queries)
        keys = self.W_k(keys)
        values = self.W_v(values)

        queries = transpose_qkv(queries, self.num_heads)
        keys = transpose_qkv(keys, self.num_heads)
        values = transpose_qkv(values, self.num_heads)

        output = self.attention(queries, keys, values, window_size)
        attn_weights = self.attention.attention_weights

        output_concat = transpose_output(output, self.num_heads)
        output_final = self.W_o(output_concat)

        return output_final





In [82]:
# 🧪 Debug
Q = torch.randn(1, 6, 8)
K = torch.randn(1, 6, 8)
V = torch.randn(1, 6, 8)

mha = MultiHeadAttention(2, d_model=8, dropout=0.0)
out = mha(Q, K, V, window_size=1)
print("Output shape:", out.shape)  # (1, 6, 8)

# window=0 should return V
out0 = mha(Q, K, V, window_size=0)
print("window=0 == V?", torch.allclose(out0, V, atol=1e-5))

Output shape: torch.Size([1, 6, 8])
window=0 == V? False


In [83]:
from torch_judge import check
check('mha')


🧪 Testing: Multi-Head Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/6] Output shape (10.1ms)
  ✅ [2/6] Uses nn.Linear with correct shapes (1.1ms)
  ✅ [3/6] Numerical correctness vs reference (4.9ms)
  ✅ [4/6] Gradient flow (6.0ms)
  ✅ [5/6] Cross-attention (seq_q != seq_k) (2.2ms)
  ✅ [6/6] Different heads give different outputs (2.9ms)
──────────────────────────────────────────────────
  🎉 All 6 tests passed! (27.2ms total)
  Progress saved. Run status() to see your dashboard.

